# Model Training for Bank Marketing Prediction

## 1. Import Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib # For saving the model
import json # For loading feature names

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score

try:
    from lazypredict.Supervised import LazyClassifier
    lazypredict_available = True
except ImportError:
    print("LazyPredict not installed. Skipping LazyPredict steps. Consider installing with: pip install lazypredict")
    lazypredict_available = False

import mlflow
import mlflow.sklearn

# Display plots inline
%matplotlib inline

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Load Data

In [ ]:
data_path = '../feature_pipeline/bank-features-selected.csv'
selected_features_path = '../feature_pipeline/selected_feature_names.json'
df = None
X = None
y = None
target_column = 'y' # As defined in feature engineering
feature_names = []

if os.path.exists(data_path):
    try:
        df = pd.read_csv(data_path)
        print(f"Loaded data from '{data_path}'. Shape: {df.shape}")
        
        if target_column in df.columns:
            X = df.drop(columns=[target_column])
            y = df[target_column]
            feature_names = X.columns.tolist()
            print(f"Features (X) shape: {X.shape}")
            print(f"Target (y) shape: {y.shape}")
            print(f"Target distribution:\n{y.value_counts(normalize=True)}")
            
            if y.isnull().any():
                print(f"\nWarning: Target variable '{target_column}' contains {y.isnull().sum()} NaN values.")
                print("Dropping rows with NaN target for model training.")
                nan_target_indices = y[y.isnull()].index
                X = X.drop(index=nan_target_indices)
                y = y.drop(index=nan_target_indices)
                feature_names = X.columns.tolist() # Update feature names if rows dropped
                print(f"Cleaned X shape: {X.shape}, Cleaned y shape: {y.shape}")
                print(f"New target distribution after NaN removal:\n{y.value_counts(normalize=True)}")
            
            if not pd.api.types.is_integer_dtype(y):
                print("\nWarning: Target variable is not integer type. Attempting conversion...")
                try:
                    y = y.astype(int)
                    print("Target variable converted to integer successfully.")
                except ValueError as ve:
                    print(f"Error converting target to int: {ve}. Check for non-numeric values.")
                    X, y = None, None 
        else:
            print(f"Error: Target column '{target_column}' not found in the loaded DataFrame.")
            X, y = None, None
            
        # Optionally load feature names from JSON if X.columns is not sufficient (e.g. if order matters and was fixed)
        if os.path.exists(selected_features_path):
             with open(selected_features_path, 'r') as f:
                 feature_names_from_file = json.load(f)
                 if set(feature_names_from_file) == set(X.columns.tolist()):
                     feature_names = feature_names_from_file # Use this if order is critical and defined in file
                     print(f"Loaded feature names from {selected_features_path}")
                 else:
                     print("Warning: Feature names from JSON do not match columns in CSV. Using columns from CSV.")
                     feature_names = X.columns.tolist()
        else:
            print(f"Warning: {selected_features_path} not found. Using feature names from CSV columns.")
            if X is not None: feature_names = X.columns.tolist()

    except Exception as e:
        print(f"Error loading data from '{data_path}': {e}")
        df, X, y = None, None, None
else:
    print(f"Error: Data file not found at '{data_path}'. Make sure the feature engineering pipeline has been run.")

## 3. Split Data

In [ ]:
X_train, X_test, y_train, y_test = None, None, None, None

if X is not None and y is not None:
    if not y.empty and len(y.unique()) > 1: 
        try:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
            print("Data split into training and testing sets.")
            print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
            print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
        except ValueError as ve_split: 
             print(f"Error during data splitting (possibly due to too few samples for a class for stratify): {ve_split}")
             print("Attempting split without stratification...")
             try:
                 X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                 print("Data split (without stratification) successful.")
             except Exception as e_split_nostrat:
                 print(f"Error during data splitting without stratification: {e_split_nostrat}")
        except Exception as e_split:
            print(f"An unexpected error during data splitting: {e_split}")
    else:
        print("Target variable 'y' is empty or has only one class. Cannot perform stratified split or meaningful training.")
else:
    print("X or y is not available. Skipping data splitting.")

## 4. Run LazyPredict (Optional - for comparison)

In [ ]:
models_summary = None
if lazypredict_available and X_train is not None and y_train is not None and X_test is not None and y_test is not None:
    if X_train.isnull().sum().sum() > 0 or y_train.isnull().sum().sum() > 0:
        print("Warning: NaNs found in training data. LazyPredict might fail or produce unreliable results.")

    print("Running LazyClassifier...")
    try:
        clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None, random_state=42)
        models, predictions = clf.fit(X_train, X_test, y_train, y_test)
        print("\nLazyClassifier Results:")
        display(models)
        models_summary = models 
    except Exception as e:
        print(f"Error running LazyClassifier: {e}")
elif not lazypredict_available:
    print("LazyPredict is not installed. Skipping this step.")
else:
    print("Training/testing data not available or not properly prepared. Skipping LazyClassifier.")

### Discussion of LazyPredict Results
LazyPredict provides a quick overview of various models. We look for high F1-score, ROC AUC, and Balanced Accuracy due to class imbalance.

## 5. MLflow Experiment Setup

In [ ]:
experiment_name = "Bank Marketing Predictions"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"MLflow experiment '{experiment_name}' created with ID: {experiment_id}")
except mlflow.exceptions.MlflowException as e:
    if "already exists" in str(e):
        experiment = mlflow.get_experiment_by_name(experiment_name)
        experiment_id = experiment.experiment_id
        print(f"MLflow experiment '{experiment_name}' already exists with ID: {experiment_id}")
    else:
        raise e
mlflow.set_experiment(experiment_name=experiment_name)

## 6. Hyperparameter Tuning and MLflow Tracking

We will select a model (e.g., RandomForestClassifier or GradientBoostingClassifier, possibly guided by LazyPredict results), define a parameter grid, and use GridSearchCV for tuning. Results will be logged with MLflow.

In [ ]:
model_to_tune_name = "RandomForestClassifier" # Default, can be changed
model_for_tuning = None
param_grid = {}

if models_summary is not None and not models_summary.empty:
    preferred_models_for_tuning = ["RandomForestClassifier", "GradientBoostingClassifier", "LogisticRegression"]
    # Example: Choose based on F1 Score from LazyPredict if available
    if 'F1 Score' in models_summary.columns:
        sorted_lazy_models = models_summary.sort_values('F1 Score', ascending=False)
        for m_name in sorted_lazy_models.index:
            if m_name in preferred_models_for_tuning:
                model_to_tune_name = m_name
                print(f"Selected '{model_to_tune_name}' for tuning based on LazyPredict F1 Score.")
                break
    elif 'ROC AUC' in models_summary.columns:
        sorted_lazy_models = models_summary.sort_values('ROC AUC', ascending=False)
        for m_name in sorted_lazy_models.index:
            if m_name in preferred_models_for_tuning:
                model_to_tune_name = m_name
                print(f"Selected '{model_to_tune_name}' for tuning based on LazyPredict ROC AUC Score.")
                break
else:
    print(f"LazyPredict summary not available or empty. Defaulting to '{model_to_tune_name}' for tuning.")

print(f"\nSetting up for {model_to_tune_name} tuning...")
if model_to_tune_name == "RandomForestClassifier":
    model_for_tuning = RandomForestClassifier(random_state=42, class_weight='balanced')
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }
elif model_to_tune_name == "GradientBoostingClassifier":
    model_for_tuning = GradientBoostingClassifier(random_state=42)
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7]
    }
elif model_to_tune_name == "LogisticRegression":
    model_for_tuning = LogisticRegression(random_state=42, class_weight='balanced', solver='liblinear', max_iter=1000)
    param_grid = {
        'C': [0.1, 1.0, 10.0],
        'penalty': ['l1', 'l2']
    }
else:
    print(f"Tuning for {model_to_tune_name} not pre-defined. Falling back to RandomForestClassifier.")
    model_to_tune_name = "RandomForestClassifier"
    model_for_tuning = RandomForestClassifier(random_state=42, class_weight='balanced')
    param_grid = {
        'n_estimators': [50, 100], # Reduced grid for fallback
        'max_depth': [10, None]
    }

best_model = None
if X_train is not None and y_train is not None and model_for_tuning is not None:
    print(f"Starting GridSearchCV for {model_to_tune_name}...")
    # Using 'f1_weighted' as it's good for imbalanced classes. 'roc_auc' is also a strong choice.
    grid_search = GridSearchCV(estimator=model_for_tuning, param_grid=param_grid, cv=3, scoring='f1_weighted', verbose=1, n_jobs=-1)
    try:
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        print(f"\nBest parameters for {model_to_tune_name}: {grid_search.best_params_}")
        print(f"Best F1_weighted score from GridSearchCV: {grid_search.best_score_:.4f}")
    except Exception as e:
        print(f"Error during GridSearchCV: {e}")
else:
    print("Training data or model for tuning not available. Skipping GridSearchCV.")

In [ ]:
if best_model is not None and X_test is not None and y_test is not None:
    with mlflow.start_run(run_name=f"Tuned {model_to_tune_name}") as run:
        run_id = run.info.run_id
        print(f"MLflow Run ID: {run_id}")
        mlflow.log_param("model_type", model_to_tune_name)
        mlflow.log_params(grid_search.best_params_)
        mlflow.log_param("features", feature_names) # Log list of feature names
        
        # Train final model with best params (already done by GridSearchCV if refit=True, which is default)
        # best_model.fit(X_train, y_train) # Not needed if GridSearchCV refit is True
        
        y_pred_tuned = best_model.predict(X_test)
        y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]
        
        # Log metrics
        accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
        f1_weighted_tuned = f1_score(y_test, y_pred_tuned, average='weighted')
        f1_macro_tuned = f1_score(y_test, y_pred_tuned, average='macro')
        f1_positive_class_tuned = f1_score(y_test, y_pred_tuned, pos_label=1) # F1 for 'yes'
        precision_weighted_tuned = precision_score(y_test, y_pred_tuned, average='weighted')
        precision_macro_tuned = precision_score(y_test, y_pred_tuned, average='macro')
        recall_weighted_tuned = recall_score(y_test, y_pred_tuned, average='weighted')
        recall_macro_tuned = recall_score(y_test, y_pred_tuned, average='macro')
        roc_auc_tuned = roc_auc_score(y_test, y_pred_proba_tuned)
        
        mlflow.log_metric("accuracy", accuracy_tuned)
        mlflow.log_metric("f1_weighted", f1_weighted_tuned)
        mlflow.log_metric("f1_macro", f1_macro_tuned)
        mlflow.log_metric("f1_positive_class", f1_positive_class_tuned)
        mlflow.log_metric("precision_weighted", precision_weighted_tuned)
        mlflow.log_metric("precision_macro", precision_macro_tuned)
        mlflow.log_metric("recall_weighted", recall_weighted_tuned)
        mlflow.log_metric("recall_macro", recall_macro_tuned)
        mlflow.log_metric("roc_auc", roc_auc_tuned)
        
        print(f"\nTuned {model_to_tune_name} Performance:")
        print(f"  Accuracy: {accuracy_tuned:.4f}")
        print(f"  F1-score (weighted): {f1_weighted_tuned:.4f}")
        print(f"  F1-score (macro): {f1_macro_tuned:.4f}")
        print(f"  F1-score (positive class 'yes'): {f1_positive_class_tuned:.4f}")
        print(f"  ROC AUC: {roc_auc_tuned:.4f}")
        
        # Log classification report as text file artifact
        report_str = classification_report(y_test, y_pred_tuned, target_names=['No (0)', 'Yes (1)'])
        with open("classification_report.txt", "w") as f:
            f.write(report_str)
        mlflow.log_artifact("classification_report.txt")
        print("\nClassification Report (Tuned Model):")
        print(report_str)
        
        # Log confusion matrix plot
        fig_cm, ax_cm = plt.subplots(figsize=(6,4))
        cm_tuned = confusion_matrix(y_test, y_pred_tuned)
        sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                    xticklabels=['Predicted No (0)', 'Predicted Yes (1)'], 
                    yticklabels=['Actual No (0)', 'Actual Yes (1)'])
        ax_cm.set_xlabel('Predicted Label')
        ax_cm.set_ylabel('True Label')
        ax_cm.set_title(f'Confusion Matrix - Tuned {model_to_tune_name}')
        mlflow.log_figure(fig_cm, "confusion_matrix_tuned.png")
        plt.show()
        
        # Log feature importance plot (if applicable)
        if hasattr(best_model, 'feature_importances_'):
            importances = best_model.feature_importances_
            feature_importance_df = pd.DataFrame({'feature': X_train.columns, 'importance': importances})
            feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False).head(15) # Top 15
            
            fig_fi, ax_fi = plt.subplots(figsize=(10, 6))
            sns.barplot(x='importance', y='feature', data=feature_importance_df, ax=ax_fi)
            ax_fi.set_title(f'Feature Importances - Tuned {model_to_tune_name}')
            plt.tight_layout()
            mlflow.log_figure(fig_fi, "feature_importances_tuned.png")
            plt.show()
            
        # Log the model
        mlflow.sklearn.log_model(best_model, f"tuned_{model_to_tune_name.lower()}_model")
        print(f"Tuned {model_to_tune_name} logged to MLflow.")
        
        # Save the best tuned model using joblib as well for direct use
        best_model_filename = 'best_tuned_model.joblib'
        joblib.dump(best_model, best_model_filename)
        print(f"Best tuned model saved locally as '{best_model_filename}'")
        
else:
    print("Best model not available from tuning, or test data not available. Skipping MLflow tracking and final model saving.")

### Discussion of Tuned Model Performance

Compare the metrics of the tuned model (Accuracy, F1-scores, Precision, Recall, ROC AUC) against the baseline model trained in step 5 and LazyPredict results. 
-   Did hyperparameter tuning improve performance, especially for the minority class ('yes')?
-   How do the F1-score (positive class), ROC AUC, and Balanced Accuracy compare?
-   The feature importance plot (if generated) can provide insights into what drives the model's predictions.

The model logged to MLflow (`tuned_{model_name}_model`) and saved locally as `best_tuned_model.joblib` is now the candidate for the inference pipeline.

## 7. Initial Model (from step 5) - MLflow Logging (Optional)

In [ ]:
# This section is for the model trained in step 5 (before tuning)
# It's useful if you want to log that initial model to MLflow for comparison
# Assuming 'final_model' and 'model_name_to_train' are from the earlier cell (step 5)

initial_model_object = None # Placeholder for the model from step 5
initial_model_name = None # Placeholder

# Attempt to load the previously saved model if it exists from step 5 logic
# This requires knowing which model was saved. Let's assume it was based on 'model_name_to_train' variable from that cell.
# The variable 'final_model' and 'model_name_to_train' from cell of Section 5 might be overwritten or out of scope.
# For simplicity, we'll re-define or ensure these are available if this cell is to be run independently.
# If this notebook is run top-to-bottom, 'final_model' and 'model_name_to_train' *might* still hold values from section 5.
# However, the variable 'model_name_to_train' was re-used for tuning. Let's assume the first trained model was saved with a generic name or its specific name.

# To make this cell runnable and demonstrate logging for the *first* model (before tuning):
# We would ideally re-run part of step 5 or load its saved artifact. 
# For now, let's assume 'final_model' and 'model_name_to_train' from that initial training are accessible.
# If 'final_model' variable from step 5 is not available here, this cell would need to reload it or retrain.

# Example: if the first model trained was RandomForestClassifier and saved as 'randomforestclassifier_model.joblib'
first_model_path = None
if 'model_name_to_train' in locals() and os.path.exists(f'{model_name_to_train.lower()}_model.joblib'):
    first_model_path = f'{model_name_to_train.lower()}_model.joblib'
    initial_model_name = model_name_to_train # This is actually the name of the model chosen in step 5
elif os.path.exists('randomforestclassifier_model.joblib'): # A common default
    first_model_path = 'randomforestclassifier_model.joblib'
    initial_model_name = 'RandomForestClassifier'

if first_model_path and os.path.exists(first_model_path):
    print(f"Loading initial model '{initial_model_name}' from {first_model_path} for MLflow logging...")
    initial_model_object = joblib.load(first_model_path)

    if initial_model_object and X_test is not None and y_test is not None:
        with mlflow.start_run(run_name=f"Initial {initial_model_name}") as run_initial:
            print(f"MLflow Run ID for initial model: {run_initial.info.run_id}")
            mlflow.log_param("model_type", initial_model_name)
            mlflow.log_param("parameters", initial_model_object.get_params())
            mlflow.log_param("features", feature_names)

            y_pred_initial = initial_model_object.predict(X_test)
            y_pred_proba_initial = initial_model_object.predict_proba(X_test)[:, 1]

            accuracy_initial = accuracy_score(y_test, y_pred_initial)
            f1_weighted_initial = f1_score(y_test, y_pred_initial, average='weighted')
            f1_macro_initial = f1_score(y_test, y_pred_initial, average='macro')
            f1_pos_initial = f1_score(y_test, y_pred_initial, pos_label=1)
            roc_auc_initial = roc_auc_score(y_test, y_pred_proba_initial)

            mlflow.log_metric("accuracy", accuracy_initial)
            mlflow.log_metric("f1_weighted", f1_weighted_initial)
            mlflow.log_metric("f1_macro", f1_macro_initial)
            mlflow.log_metric("f1_positive_class", f1_pos_initial)
            mlflow.log_metric("roc_auc", roc_auc_initial)
            
            print(f"\nInitial {initial_model_name} Performance (for MLflow logging):")
            print(f"  Accuracy: {accuracy_initial:.4f}, F1 (weighted): {f1_weighted_initial:.4f}, ROC AUC: {roc_auc_initial:.4f}")

            mlflow.sklearn.log_model(initial_model_object, f"initial_{initial_model_name.lower()}_model")
            print(f"Initial {initial_model_name} logged to MLflow.")
    else:
        print("Initial model from step 5 not available or test data missing. Skipping MLflow logging for it.")
else:
    print("Path to initial model from step 5 not determined or file does not exist. Skipping MLflow logging for it.")